In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# -------------------------------
# 1) Construct a sample dataset
# -------------------------------
df = pd.DataFrame({
    "EPS": [2.1, 2.3, 2.5, np.nan, 2.8, 3.0, 3.2],
    "Sector": ["Banking", "IT", "IT", "Banking", "Auto", "Auto", "IT"],
    "PE_Ratio": [12, 25, 30, 15, 18, 20, 28]
})

# -------------------------------
# 2) Handle missing values
# -------------------------------
df["EPS"] = df["EPS"].fillna(df["EPS"].mean())   # fill with mean

# -------------------------------
# 3) One-hot encoding
# -------------------------------
df = pd.get_dummies(df, columns=["Sector"], drop_first=True)

# -------------------------------
# 4) Create target variable
# Next EPS = shift(-1)
# % change = (next - current)/current
# -------------------------------
df["Next_EPS"] = df["EPS"].shift(-1)
df["Target"] = (df["Next_EPS"] - df["EPS"]) / df["EPS"]

# Drop last row (no next EPS available)
df = df.dropna()

# -------------------------------
# 5) Correlation analysis
# -------------------------------
corrs = df.corr()["Target"].sort_values()
print("Top positive correlations:\n", corrs.tail(10))
print("Top negative correlations:\n", corrs.head(10))

# -------------------------------
# 6) Feature selection
# (keep only top/bottom correlated features)
# -------------------------------
selected_features = corrs.index[-10:].tolist() + corrs.index[:10].tolist()
selected_features = [f for f in selected_features if f not in ["Target"]]

X = df[selected_features]
y = df["Target"]

# -------------------------------
# 7) Train/test split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# -------------------------------
# 8) Train model
# -------------------------------
model = LinearRegression()
model.fit(X_train, y_train)

# -------------------------------
# 9) Predict
# -------------------------------
y_pred = model.predict(X_test)
print("Predictions:", y_pred)
print("Actual:", list(y_test))

Top positive correlations:
 EPS              -0.703742
Next_EPS         -0.646100
PE_Ratio         -0.303007
Sector_IT         0.033562
Sector_Banking    0.157276
Target            1.000000
Name: Target, dtype: float64
Top negative correlations:
 EPS              -0.703742
Next_EPS         -0.646100
PE_Ratio         -0.303007
Sector_IT         0.033562
Sector_Banking    0.157276
Target            1.000000
Name: Target, dtype: float64
Predictions: [0.06607162 0.06860479]
Actual: [0.09523809523809511, 0.08695652173913052]
